# Quantum VQC Classifier

Binary classification on the Iris dataset using a Variational Quantum Classifier (VQC)
built with `qiskit-machine-learning`.

This notebook trains the quantum model and saves results to `vqc_results.json`  
for later use in `comparison/comparison.ipynb`.

**How a VQC works (briefly):**  
A VQC is a parameterized quantum circuit split into two parts:
1. **Feature map**: encodes classical input data into quantum states (angles of rotation gates).
2. **Ansatz**: a trainable circuit whose gate angles are optimized during training.

The output is the probability of measuring a certain qubit state, which is mapped to a class label.
Training adjusts the ansatz parameters to minimize a loss function, exactly like gradient descent
in a classical neural network.

**Configuration variable:** `N_FEATURES` controls how many features (and qubits) are used.  
For the 4-feature version, set `N_FEATURES = 4` and results save to `vqc_results_4f.json`.

**Hardware:** Training runs on the Aer simulator. Set `RUN_ON_REAL_HW = True` to evaluate
the trained model on IBM Quantum hardware (requires a valid IBM account configured).

## Imports

In [ ]:
import time
import json
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import Sampler as AerSampler

from qiskit_machine_learning.algorithms import VQC

from scipy.optimize import minimize

## Configuration

In [ ]:
N_FEATURES      = 2          # 2 or 4 — must match svm.ipynb
TEST_SIZE       = 0.2        # must match svm.ipynb
RANDOM_SEED     = 42         # must match svm.ipynb
MAX_ITER        = 100        # optimizer iterations (increase for better convergence)
RUN_ON_REAL_HW  = False      # set True to evaluate on IBM Quantum after training

RESULTS_FILE = "vqc_results.json" if N_FEATURES == 2 else "vqc_results_4f.json"

print(f"Configuration: N_FEATURES={N_FEATURES}, MAX_ITER={MAX_ITER}, RUN_ON_REAL_HW={RUN_ON_REAL_HW}")
print(f"Results will be saved to: {RESULTS_FILE}")

## Dataset

Identical preprocessing to `svm.ipynb` — same seed, same split, same scaling.
Both models must receive the exact same train/test data for a fair comparison.

In [ ]:
iris = load_iris()

mask   = iris.target != 0
X_full = iris.data[mask]
y_full = iris.target[mask] - 1      # remap to {0, 1}

if N_FEATURES == 2:
    X = X_full[:, 2:4]
    feature_names = ["petal length", "petal width"]
else:
    X = X_full
    feature_names = list(iris.feature_names)

scaler   = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Scale to [0, π] so features span the full range of rotation gates
X_scaled_pi = X_scaled * np.pi

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_pi, y_full, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_full
)

print(f"Total samples : {len(X_scaled_pi)}")
print(f"Train samples : {len(X_train)}")
print(f"Test samples  : {len(X_test)}")
print(f"Features used : {feature_names}")

## Quantum Algorithm — VQC

### Circuit architecture

**Feature map — `ZZFeatureMap`**  
Encodes each input vector into a quantum state using rotation gates (Ry, Rz) and
entangling ZZ interactions. With `reps=2`, the encoding applies twice, increasing
expressiveness. The number of qubits equals `N_FEATURES`.

**Ansatz — `RealAmplitudes`**  
A hardware-efficient variational circuit with Ry rotations and CNOT entangling layers.
With `reps=3`, the circuit has enough parameters to represent complex decision boundaries.
These are the trainable parameters.

**Optimizer — COBYLA**  
A gradient-free optimizer. We use it because computing exact gradients on a quantum
circuit requires additional circuit evaluations (parameter-shift rule), which is slow
in simulation. COBYLA converges well on small problems.

In [ ]:
n_qubits    = N_FEATURES
feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)
ansatz      = RealAmplitudes(num_qubits=n_qubits, reps=3)

print(f"Number of qubits         : {n_qubits}")
print(f"Feature map depth (reps) : 2")
print(f"Ansatz depth (reps)      : 3")
print(f"Trainable parameters     : {ansatz.num_parameters}")

# Draw the feature map and ansatz
print("\n--- Feature Map ---")
print(feature_map.decompose().draw(output="text", fold=-1))
print("\n--- Ansatz ---")
print(ansatz.draw(output="text", fold=-1))

## Training on Aer Simulator

In [ ]:
# Track loss evolution during training
loss_history = []

def callback(weights, obj_func_eval):
    loss_history.append(obj_func_eval)
    if len(loss_history) % 10 == 0:
        print(f"  Iteration {len(loss_history):>4} | Loss: {obj_func_eval:.4f}")

sampler = AerSampler()

vqc = VQC(
    sampler      = sampler,
    feature_map  = feature_map,
    ansatz       = ansatz,
    optimizer    = {"maxiter": MAX_ITER},   # COBYLA by default in VQC
    callback     = callback,
)

print(f"Training VQC on Aer simulator ({MAX_ITER} max iterations)...")
t0 = time.perf_counter()
vqc.fit(X_train, y_train)
train_time_sim = time.perf_counter() - t0

print(f"\nTraining completed in {train_time_sim:.2f} s")
print(f"Final loss: {loss_history[-1]:.4f}")

## Evaluation on Simulator

In [ ]:
y_pred_train_sim = vqc.predict(X_train)
y_pred_test_sim  = vqc.predict(X_test)

acc_train_sim = accuracy_score(y_train, y_pred_train_sim)
acc_test_sim  = accuracy_score(y_test,  y_pred_test_sim)

print(f"Train accuracy (simulator): {acc_train_sim:.4f}")
print(f"Test accuracy  (simulator): {acc_test_sim:.4f}")
print()
print(classification_report(y_test, y_pred_test_sim,
                             target_names=["Versicolor", "Virginica"]))

## Evaluation on IBM Quantum Hardware (optional)

The model is **trained** on the simulator and **evaluated** on real hardware.
This avoids the prohibitive queue time of running hundreds of training iterations
on real hardware, while still showing how noise affects inference.

Set `RUN_ON_REAL_HW = True` in the Configuration cell to enable this section.

In [ ]:
acc_test_real    = None
y_pred_test_real = None
backend_name     = None
circuit_depth_real = None

if RUN_ON_REAL_HW:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RealSampler
    from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False,
                                  min_num_qubits=n_qubits)
    backend_name = backend.name
    print(f"Using backend: {backend_name}")

    # Build the full VQC inference circuit with trained weights bound
    # and transpile it to the real hardware topology
    real_sampler = RealSampler(backend)

    vqc_real = VQC(
        sampler     = real_sampler,
        feature_map = feature_map,
        ansatz      = ansatz,
    )
    # Transfer trained weights
    vqc_real.weights = vqc.weights

    print("Running inference on real hardware...")
    t0 = time.perf_counter()
    y_pred_test_real = vqc_real.predict(X_test)
    infer_time_real  = time.perf_counter() - t0

    acc_test_real = accuracy_score(y_test, y_pred_test_real)
    print(f"Test accuracy (real HW): {acc_test_real:.4f}")
    print(f"Inference time        : {infer_time_real:.2f} s")
else:
    print("RUN_ON_REAL_HW is False — skipping real hardware evaluation.")
    print("Set RUN_ON_REAL_HW = True in the Configuration cell to enable.")

## Plots

In [ ]:
# --- Loss curve ---
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, color="tab:blue")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title(f"VQC Training Loss ({N_FEATURES} qubits, Aer simulator)")
plt.grid(True)
plt.tight_layout()
plt.savefig(f"vqc_loss_curve{'_4f' if N_FEATURES == 4 else ''}.png", dpi=150)
plt.show()

In [ ]:
# --- Confusion matrix (simulator) ---
cm = confusion_matrix(y_test, y_pred_test_sim)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Versicolor", "Virginica"])
disp.plot(colorbar=False)
plt.title(f"VQC — Confusion Matrix, Simulator ({N_FEATURES} qubits)")
plt.tight_layout()
plt.savefig(f"vqc_confusion_matrix_sim{'_4f' if N_FEATURES == 4 else ''}.png", dpi=150)
plt.show()

In [ ]:
# --- Decision boundary (only for 2 features) ---
if N_FEATURES == 2:
    h = 0.02
    xx, yy = np.meshgrid(np.arange(0, np.pi + h, h), np.arange(0, np.pi + h, h))
    grid   = np.c_[xx.ravel(), yy.ravel()]
    Z      = vqc.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    scatter = plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train,
                          cmap="coolwarm", edgecolors="k", s=40, label="Train")
    plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test,
                cmap="coolwarm", edgecolors="r", s=60, marker="^", label="Test")
    plt.colorbar(scatter, ticks=[0, 1], label="Class")
    plt.xlabel("petal length (scaled × π)")
    plt.ylabel("petal width  (scaled × π)")
    plt.title("VQC — Decision Boundary (2 qubits, Aer simulator)")
    plt.legend()
    plt.tight_layout()
    plt.savefig("vqc_decision_boundary.png", dpi=150)
    plt.show()
else:
    print("Decision boundary plot skipped (only available for N_FEATURES=2).")

## Save Results

In [ ]:
results = {
    # --- Experiment metadata ---
    "model"              : "VQC",
    "n_features"         : N_FEATURES,
    "n_qubits"           : n_qubits,
    "feature_names"      : feature_names,
    "n_train"            : int(len(X_train)),
    "n_test"             : int(len(X_test)),
    "random_seed"        : RANDOM_SEED,
    # --- Circuit architecture ---
    "feature_map"        : "ZZFeatureMap",
    "feature_map_reps"   : 2,
    "ansatz"             : "RealAmplitudes",
    "ansatz_reps"        : 3,
    "n_parameters"       : int(ansatz.num_parameters),
    "optimizer"          : "COBYLA",
    "max_iter"           : MAX_ITER,
    # --- Timing ---
    "train_time_sim_s"   : float(train_time_sim),
    # --- Accuracy (simulator) ---
    "accuracy_test_sim"  : float(acc_test_sim),
    "accuracy_train_sim" : float(acc_train_sim),
    # --- Loss history ---
    "loss_history"       : [float(l) for l in loss_history],
    # --- Predictions (simulator) ---
    "y_test"             : y_test.tolist(),
    "y_pred_test_sim"    : y_pred_test_sim.tolist(),
    # --- Real hardware (filled only if RUN_ON_REAL_HW=True) ---
    "real_hw_run"        : RUN_ON_REAL_HW,
    "backend_name"       : backend_name,
    "accuracy_test_real" : float(acc_test_real) if acc_test_real is not None else None,
    "y_pred_test_real"   : y_pred_test_real.tolist() if y_pred_test_real is not None else None,
}

with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {RESULTS_FILE}")
# Print summary without the long lists
summary_keys = [k for k in results if k not in ("loss_history", "y_test", "y_pred_test_sim", "y_pred_test_real")]
print(json.dumps({k: results[k] for k in summary_keys}, indent=2))